In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping

torch.set_float32_matmul_precision('medium')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"GPU: {torch.cuda.get_device_name(0)}")

# Carregar dados
df_series = pd.read_parquet('/home/valentim/divea/data/processed/series_semanais.parquet')
print(f"Series carregadas: {df_series.shape}")

GPU: NVIDIA GeForce RTX 5060 Ti
Series carregadas: (382, 4)


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/pytorch_forecasting/models/base_model.py:27: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
def criar_dataset_tft(serie, janela=16, split=0.7):
    df = serie.reset_index()
    df.columns = ['data', 'casos']
    df['time_idx'] = range(len(df))
    df['grupo'] = 'parana'
    df['casos'] = df['casos'].astype(float)
    df['semana'] = df['data'].dt.isocalendar().week.astype(str)
    df['mes'] = df['data'].dt.month.astype(str)
    
    SPLIT = int(len(df) * split)
    
    treino = TimeSeriesDataSet(
        df[df['time_idx'] <= SPLIT],
        time_idx='time_idx', target='casos', group_ids=['grupo'],
        min_encoder_length=janela, max_encoder_length=janela,
        min_prediction_length=4, max_prediction_length=4,
        time_varying_known_categoricals=['semana', 'mes'],
        time_varying_unknown_reals=['casos'],
        target_normalizer=GroupNormalizer(groups=['grupo'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    )
    validacao = TimeSeriesDataSet.from_dataset(treino, df, predict=False, stop_randomization=True)
    train_loader = treino.to_dataloader(train=True, batch_size=32, num_workers=0)
    val_loader = validacao.to_dataloader(train=False, batch_size=32, num_workers=0)
    return treino, train_loader, val_loader, df

def treinar_tft(treino, train_loader, val_loader, epochs=80):
    model = TemporalFusionTransformer.from_dataset(
        treino, learning_rate=0.03, hidden_size=32,
        attention_head_size=2, dropout=0.1, hidden_continuous_size=16,
        loss=QuantileLoss(), optimizer='adam', log_interval=-1,
        reduce_on_plateau_patience=4,
    )
    early_stop = EarlyStopping(monitor='val_loss', patience=10, mode='min')
    trainer = Trainer(
        max_epochs=epochs, accelerator='gpu', devices=1,
        callbacks=[early_stop], enable_progress_bar=False,
        log_every_n_steps=1, logger=False,
    )
    trainer.fit(model, train_loader, val_loader)
    return model, trainer

def avaliar_tft(model, val_loader):
    model.eval()
    all_preds, all_reals = [], []
    with torch.no_grad():
        for batch in val_loader:
            x, y = batch
            out = model(x)
            all_preds.append(out['prediction'].cpu().numpy())
            all_reals.append(y[0].cpu().numpy())
    y_pred = np.concatenate(all_preds, axis=0)
    y_real = np.concatenate(all_reals, axis=0)
    for s in range(4):
        mae = mean_absolute_error(y_real[:, s], y_pred[:, s, 3])
        print(f"  Semana +{s+1} MAE: {mae:.1f}")
    print(f"  Media real: {y_real.mean():.1f}")
    return y_pred, y_real

print("Funcoes definidas.")

Funcoes definidas.


In [3]:
# SRAG Total
print("=== SRAG Total ===")
treino_srag, tl_srag, vl_srag, _ = criar_dataset_tft(df_series['casos'], janela=16)
tft_srag, tr_srag = treinar_tft(treino_srag, tl_srag, vl_srag)
print(f"Epocas: {tr_srag.current_epoch} | Val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)
torch.save(tft_srag.state_dict(), '/home/valentim/divea/models/tft_srag_total.pt')

# Influenza
print("\n=== Influenza ===")
serie_flu = df_series['influenza'].copy()
serie_flu = serie_flu[(serie_flu.index < '2020-01-01') | (serie_flu.index >= '2022-01-01')]
treino_flu, tl_flu, vl_flu, _ = criar_dataset_tft(serie_flu, janela=8)
tft_flu, tr_flu = treinar_tft(treino_flu, tl_flu, vl_flu)
print(f"Epocas: {tr_flu.current_epoch} | Val loss: {tr_flu.callback_metrics.get('val_loss'):.2f}")
y_pred_flu, y_real_flu = avaliar_tft(tft_flu, vl_flu)
torch.save(tft_flu.state_dict(), '/home/valentim/divea/models/tft_influenza.pt')

# VSR
print("\n=== VSR ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=8)
tft_vsr, tr_vsr = treinar_tft(treino_vsr, tl_vsr, vl_vsr)
print(f"Epocas: {tr_vsr.current_epoch} | Val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)
torch.save(tft_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')

print("\n=== Todos os modelos salvos ===")

=== SRAG Total ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proj

Epocas: 27 | Val loss: 157.04
  Semana +1 MAE: 143.2
  Semana +2 MAE: 136.7
  Semana +3 MAE: 137.6
  Semana +4 MAE: 151.4
  Media real: 1012.8

=== Influenza ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proj

Epocas: 57 | Val loss: 9.32
  Semana +1 MAE: 10.1
  Semana +2 MAE: 10.6
  Semana +3 MAE: 11.2
  Semana +4 MAE: 12.1
  Media real: 35.3

=== VSR ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epocas: 23 | Val loss: 16.57
  Semana +1 MAE: 40.5
  Semana +2 MAE: 40.5
  Semana +3 MAE: 40.5
  Semana +4 MAE: 40.5
  Media real: 40.5

=== Todos os modelos salvos ===


In [4]:
print(f"y_pred_srag shape: {y_pred_srag.shape}")
print(f"y_real_srag shape: {y_real_srag.shape}")
print(f"Amostra real: {y_real_srag[0]}")
print(f"Amostra pred: {y_pred_srag[0, :, 3]}")


y_pred_srag shape: (363, 4, 7)
y_real_srag shape: (363, 4)
Amostra real: [199. 238. 235. 230.]
Amostra pred: [544.4517  354.68347 442.18958 586.0586 ]


In [5]:
# Reavaliar cada modelo separadamente
print("=== SRAG Total ===")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)

print("\n=== Influenza ===")
y_pred_flu, y_real_flu = avaliar_tft(tft_flu, vl_flu)

print("\n=== VSR ===")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)

=== SRAG Total ===
  Semana +1 MAE: 143.2
  Semana +2 MAE: 136.7
  Semana +3 MAE: 137.6
  Semana +4 MAE: 151.4
  Media real: 1012.8

=== Influenza ===
  Semana +1 MAE: 10.1
  Semana +2 MAE: 10.6
  Semana +3 MAE: 11.2
  Semana +4 MAE: 12.1
  Media real: 35.3

=== VSR ===
  Semana +1 MAE: 40.5
  Semana +2 MAE: 40.5
  Semana +3 MAE: 40.5
  Semana +4 MAE: 40.5
  Media real: 40.5


In [6]:
print(f"SRAG val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
print(f"VSR val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
print(f"SRAG epocas: {tr_srag.current_epoch}")
print(f"VSR epocas: {tr_vsr.current_epoch}")


SRAG val loss: 157.04
VSR val loss: 16.57
SRAG epocas: 27
VSR epocas: 23


In [7]:
print("=== SRAG Total ===")
treino_srag, tl_srag, vl_srag, _ = criar_dataset_tft(df_series['casos'], janela=16)
tft_srag, tr_srag = treinar_tft(treino_srag, tl_srag, vl_srag, epochs=120)
print(f"Epocas: {tr_srag.current_epoch} | Val loss: {tr_srag.callback_metrics.get('val_loss'):.2f}")
y_pred_srag, y_real_srag = avaliar_tft(tft_srag, vl_srag)
torch.save(tft_srag.state_dict(), '/home/valentim/divea/models/tft_srag_total.pt')
print("SRAG salvo.")


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proj

=== SRAG Total ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epocas: 103 | Val loss: 47.68
  Semana +1 MAE: 61.9
  Semana +2 MAE: 66.0
  Semana +3 MAE: 66.0
  Semana +4 MAE: 67.8
  Media real: 1012.8
SRAG salvo.


In [8]:
print("=== VSR ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=8)
tft_vsr, tr_vsr = treinar_tft(treino_vsr, tl_vsr, vl_vsr, epochs=120)
print(f"Epocas: {tr_vsr.current_epoch} | Val loss: {tr_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(tft_vsr, vl_vsr)
torch.save(tft_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')
print("VSR salvo.")

/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proj

=== VSR ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epocas: 40 | Val loss: 16.02
  Semana +1 MAE: 40.5
  Semana +2 MAE: 40.5
  Semana +3 MAE: 40.5
  Semana +4 MAE: 40.5
  Media real: 40.5
VSR salvo.


In [9]:
print(df_series['vsr'].describe())
print(f"\nZeros: {(df_series['vsr'] == 0).sum()}")
print(f"Valores > 0: {(df_series['vsr'] > 0).sum()}")

count    382.000000
mean      39.458115
std       59.327837
min        0.000000
25%        3.000000
50%       14.000000
75%       49.750000
max      340.000000
Name: vsr, dtype: float64

Zeros: 40
Valores > 0: 342


In [10]:
print("=== VSR (janela 16, lr menor) ===")
treino_vsr, tl_vsr, vl_vsr, _ = criar_dataset_tft(df_series['vsr'], janela=16)

model_vsr = TemporalFusionTransformer.from_dataset(
    treino_vsr, learning_rate=0.01, hidden_size=64,
    attention_head_size=2, dropout=0.1, hidden_continuous_size=16,
    loss=QuantileLoss(), optimizer='adam', log_interval=-1,
    reduce_on_plateau_patience=6,
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, mode='min')
trainer_vsr = Trainer(
    max_epochs=150, accelerator='gpu', devices=1,
    callbacks=[early_stop], enable_progress_bar=False,
    log_every_n_steps=1, logger=False,
)
trainer_vsr.fit(model_vsr, tl_vsr, vl_vsr)
print(f"Epocas: {trainer_vsr.current_epoch} | Val loss: {trainer_vsr.callback_metrics.get('val_loss'):.2f}")
y_pred_vsr, y_real_vsr = avaliar_tft(model_vsr, vl_vsr)
torch.save(model_vsr.state_dict(), '/home/valentim/divea/models/tft_vsr.pt')
print("VSR salvo.")

/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/proj

=== VSR (janela 16, lr menor) ===


/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/valentim/miniconda3/envs/divea/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Epocas: 47 | Val loss: 12.12
  Semana +1 MAE: 10.3
  Semana +2 MAE: 10.9
  Semana +3 MAE: 11.2
  Semana +4 MAE: 11.8
  Media real: 40.7
VSR salvo.


In [11]:
print("=== Comparativo Final TFT vs LSTM ===")
print(f"{'Modelo':<20} {'TFT MAE+1':>10} {'TFT MAE+4':>10} {'LSTM MAE+1':>11} {'LSTM MAE+4':>11}")
print("-" * 65)
print(f"{'SRAG Total':<20} {'61.9':>10} {'67.8':>10} {'89.9':>11} {'175.8':>11}")
print(f"{'Influenza':<20} {'10.1':>10} {'12.1':>10} {'21.7':>11} {'44.9':>11}")
print(f"{'VSR':<20} {'10.3':>10} {'11.8':>10} {'19.2':>11} {'31.2':>11}")


=== Comparativo Final TFT vs LSTM ===
Modelo                TFT MAE+1  TFT MAE+4  LSTM MAE+1  LSTM MAE+4
-----------------------------------------------------------------
SRAG Total                 61.9       67.8        89.9       175.8
Influenza                  10.1       12.1        21.7        44.9
VSR                        10.3       11.8        19.2        31.2
